In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

# Updated path provided by the user
file_path = '/content/drive/MyDrive/Colab Notebooks/CL-NLP - Lab/lab-6/glove_corpus.txt'

if os.path.exists(file_path):
    print("File found! Reading the first few lines...")
    with open(file_path, 'r', encoding='utf-8') as f:
        for _ in range(10):
            line = f.readline()
            if not line: break
            print(line.strip())
else:
    print(f"File not found at {file_path}. Please verify the path on your Google Drive.")

File found! Reading the first few lines...
The king rules the kingdom.
The queen rules the kingdom.
The king is a powerful man.
The queen is a wise woman.
The man is strong and brave.
The woman is intelligent and kind.
The king and queen rule together.
The man and woman live in the kingdom.
The prince is the son of the king.
The princess is the daughter of the queen.


### Implement Vocabulary Building and Word Embeddings using GloVe
We will parse the corpus, build a vocabulary, and prepare to utilize word representations.

In [ ]:
from collections import Counter
import numpy as np

def build_vocab(file_path):
    vocab = Counter()
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = line.strip().lower().split()
            vocab.update(tokens)
    return vocab

vocab = build_vocab(file_path)
print(f"Vocabulary Size: {len(vocab)}")
print("Most common words:", vocab.most_common(10))

Vocabulary Size: 73
Most common words: [('the', 27), ('a', 12), ('and', 8), ('is', 6), ('in', 6), ('king', 4), ('queen', 4), ('kingdom.', 3), ('rules', 2), ('man', 2)]


### Part 1: Implementing GloVe From Scratch
We will construct a co-occurrence matrix, define the weighting function and loss, and optimize word vectors using gradient descent (Adam) in PyTorch to obtain custom embeddings from our corpus.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from collections import Counter, defaultdict

# 1. Clean tokens and build vocabulary
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read().lower().replace('.', ' .').replace(',', ' ,').split()

vocab = list(set(text))
word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for i, word in enumerate(vocab)}
vocab_size = len(vocab)
print(f"Cleaned Vocabulary Size: {vocab_size}")

Cleaned Vocabulary Size: 69


In [ ]:
# 2. Build Co-occurrence Matrix
window_size = 4
co_occurrence = defaultdict(float)

for i in range(len(text)):
    target_id = word_to_id[text[i]]
    start = max(0, i - window_size)
    end = min(len(text), i + window_size + 1)
    for j in range(start, end):
        if i == j:
            continue
        context_id = word_to_id[text[j]]
        # Weight is inversely proportional to distance
        distance = abs(i - j)
        co_occurrence[(target_id, context_id)] += 1.0 / distance

# Convert to lists for PyTorch dataset
x_i, x_j, co_occur_val = [], [], []
for (i, j), val in co_occurrence.items():
    x_i.append(i)
    x_j.append(j)
    co_occur_val.append(val)

x_i = torch.tensor(x_i, dtype=torch.long)
x_j = torch.tensor(x_j, dtype=torch.long)
co_occur_val = torch.tensor(co_occur_val, dtype=torch.float32)

In [ ]:
# 3. Define GloVe Model in PyTorch
class GloVe(nn.Module):
    def __init__(self, vocab_size, embedding_dim=50):
        super(GloVe, self).__init__()
        self.w_target = nn.Embedding(vocab_size, embedding_dim)
        self.w_context = nn.Embedding(vocab_size, embedding_dim)
        self.b_target = nn.Embedding(vocab_size, 1)
        self.b_context = nn.Embedding(vocab_size, 1)

        # Initialize weights
        init_range = 0.5 / embedding_dim
        self.w_target.weight.data.uniform_(-init_range, init_range)
        self.w_context.weight.data.uniform_(-init_range, init_range)
        self.b_target.weight.data.zero_()
        self.b_context.weight.data.zero_()

    def forward(self, i, j):
        w_t = self.w_target(i)
        w_c = self.w_context(j)
        b_t = self.b_target(i).squeeze(1)
        b_c = self.b_context(j).squeeze(1)
        return (w_t * w_c).sum(dim=1) + b_t + b_c

# Weighting function components
x_max = 100
alpha = 0.75
def weighting_fn(x):
    return torch.where(x < x_max, (x / x_max) ** alpha, torch.ones_like(x))

# Instantiate model
embedding_dim = 25
model = GloVe(vocab_size, embedding_dim)
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 4. Training loop
epochs = 200
for epoch in range(epochs):
    optimizer.zero_grad()
    predictions = model(x_i, x_j)
    targets = torch.log(co_occur_val)
    weights = weighting_fn(co_occur_val)
    loss = (weights * (predictions - targets) ** 2).mean()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 40 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

Epoch 40/200 | Loss: 0.0016
Epoch 80/200 | Loss: 0.0001
Epoch 120/200 | Loss: 0.0000
Epoch 160/200 | Loss: 0.0000
Epoch 200/200 | Loss: 0.0000


In [ ]:
# 5. Word Similarity Search
word_vectors = model.w_target.weight.data + model.w_context.weight.data
word_norms = word_vectors / word_vectors.norm(dim=1, keepdim=True)

def find_similar_words(target_word, top_k=5):
    if target_word not in word_to_id:
        return f"'{target_word}' not in corpus vocabulary."
    target_idx = word_to_id[target_word]
    target_vector = word_norms[target_idx]

    similarities = torch.matmul(word_norms, target_vector)
    values, indices = torch.topk(similarities, top_k + 1)

    results = []
    for val, idx in zip(values[1:], indices[1:]): # skip the target word itself
        results.append((id_to_word[idx.item()], val.item()))
    return results

print("Words most similar to 'king' (Scratch implementation):")
for word, sim in find_similar_words('king', top_k=5):
    print(f"{word}: {sim:.4f}")

Words most similar to 'king' (Scratch implementation):
the: 0.8458
queen: 0.8414
.: 0.6780
is: 0.5559
cat: 0.5208


### Part 2: Implementing GloVe Using a Pretrained Model (Gensim)
Now, we load a pre-trained GloVe model (`glove-wiki-gigaword-50` via Gensim API) and evaluate similarity with 'king'.

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 36.2 MB/s  0:00:00


In [ ]:
import gensim.downloader as api

# Load pretrained GloVe word vectors
print("Loading pretrained GloVe vectors (glove-wiki-gigaword-50)... This might take a minute.")
pretrained_glove = api.load("glove-wiki-gigaword-50")

# Query similarity
print("\nWords most similar to 'king' (Pretrained Model):")
similar_words = pretrained_glove.most_similar('king', topn=5)
for word, similarity in similar_words:
    print(f"{word}: {similarity:.4f}")

Loading pretrained GloVe vectors (glove-wiki-gigaword-50)... This might take a minute.
[==================================================] 100.0% 66.0/66.0MB downloaded

Words most similar to 'king' (Pretrained Model):
prince: 0.8236
queen: 0.7839
ii: 0.7746
emperor: 0.7736
son: 0.7667


### Part 3: Statistical N-gram Language Model from Scratch
We will implement a Bigram language model to learn transitions between words from the given corpus, compute probability distributions, and predict the next word in a sequence.

In [ ]:
from collections import Counter, defaultdict

# 1. Prepare the corpus
corpus = [
    "natural language processing is a branch of artificial intelligence.",
    "natural language processing enables computers to understand human language.",
    "machine learning is a branch of artificial intelligence.",
    "deep learning is a part of machine learning.",
    "language models learn patterns from text data.",
    "a language model predicts the next word in a sentence.",
    "natural language processing uses language models.",
    "artificial intelligence includes machine learning and deep learning."
]

# Tokenize and add sentence boundary markers (optional, but helpful for structure)
tokenized_corpus = []
for sentence in corpus:
    # Clean and split into tokens
    tokens = ["<s>"] + sentence.lower().replace('.', '').split() + ["</s>"]
    tokenized_corpus.append(tokens)

print("Tokenized first sentence:", tokenized_corpus[0])

Tokenized first sentence: ['<s>', 'natural', 'language', 'processing', 'is', 'a', 'branch', 'of', 'artificial', 'intelligence', '</s>']


In [ ]:
# 2. Calculate Unigram and Bigram Frequencies
unigram_counts = Counter()
bigram_counts = defaultdict(Counter)

for sentence in tokenized_corpus:
    for i in range(len(sentence)):
        unigram_counts[sentence[i]] += 1
        if i < len(sentence) - 1:
            current_word = sentence[i]
            next_word = sentence[i+1]
            bigram_counts[current_word][next_word] += 1

print("Total unique unigrams:", len(unigram_counts))
print("Words following 'learning':", dict(bigram_counts['learning']))

Total unique unigrams: 36
Words following 'learning': {'is': 2, '</s>': 2, 'and': 1}


In [ ]:
# 3. Predict the next word using Maximum Likelihood Estimation (MLE) with Laplace Smoothing
def predict_next_word(context_word):
    context_word = context_word.lower().strip()
    if context_word not in unigram_counts:
        return "<UNK> (Context word not found in vocabulary)"

    # Get raw candidates and apply Laplace (plus-one) smoothing
    candidates = bigram_counts[context_word]
    vocab_size = len(unigram_counts)

    probabilities = {}
    # Check all vocabulary words as potential candidates to handle smoothing properly
    for word in unigram_counts:
        if word == "<s>":
            continue
        count_transition = candidates[word]
        # Laplace formula: P(w_i | w_i-1) = (count(w_i-1, w_i) + 1) / (count(w_i-1) + V)
        probabilities[word] = (count_transition + 1) / (unigram_counts[context_word] + vocab_size)

    # Sort candidates by probability
    sorted_predictions = sorted(probabilities.items(), key=lambda item: item[1], reverse=True)
    return sorted_predictions[:3]

# Let's test predictions
test_words = ['learning', 'natural', 'artificial', 'language']
for word in test_words:
    predictions = predict_next_word(word)
    print(f"\nTop predicted next words after '{word}':")
    for next_w, prob in predictions:
        print(f"  - '{next_w}': {prob:.4f}")


Top predicted next words after 'learning':
  - 'is': 0.0732
  - '</s>': 0.0732
  - 'and': 0.0488

Top predicted next words after 'natural':
  - 'language': 0.1026
  - 'natural': 0.0256
  - 'processing': 0.0256

Top predicted next words after 'artificial':
  - 'intelligence': 0.1026
  - 'natural': 0.0256
  - 'language': 0.0256

Top predicted next words after 'language':
  - 'processing': 0.0930
  - 'models': 0.0698
  - '</s>': 0.0465
